In [ ]:
etherpad_url = "http://localhost:9001/"
migration_pad_title = "Solr Migration Test Pad"
default_result_path = None
close_on_fail = False
transition_timeout = 10000

# Solr Volume Migration Test - Phase 3: Recovery after wiping the volume

With an empty index, ep_search reindexes all pads at Etherpad startup.
The pad created in phase 1 must be searchable again.

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## The pad is searchable again.

In [ ]:
import asyncio

index_page = None

async def _step(page):
    await page.goto(etherpad_url)

    await expect(page.locator(".hashview-search-box")).to_be_editable()
    for attempt in range(20):
        await page.reload()
        if await page.locator(f'text="{migration_pad_title}"').count() > 0:
            break
        await asyncio.sleep(3)
    await expect(page.locator(f'text="{migration_pad_title}"')).to_be_visible()

    global index_page
    index_page = page

await run_pw(_step)

## Search by the title works.

In [ ]:
async def _step(page):
    await index_page.locator(".hashview-search-box").fill(f'"{migration_pad_title}"')
    await index_page.keyboard.press("Enter")
    await expect(index_page.locator('.hash-link')).to_have_count(1, timeout=transition_timeout)

await run_pw(_step)

Clean up

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}